# Lesson 8: Enriching the Listings - String, Date & UDF Magic

---

## The Staging Analogy

In real estate, **staging a property** transforms a raw, empty house into something buyers can imagine living in.
Raw data is the same way — it tells only half the story.

Today you'll transform raw fields into **meaningful features**:

- Turning `year_built` into age categories like *Brand New* or *Vintage*
- Extracting insights from listing dates (which day of the week do agents list the most?)
- Writing custom Python functions (UDFs) for logic that built-in functions can't handle

By the end of this lesson your listings DataFrame will be clean, rich, and ready for analysis.

### What you'll learn
| Topic | Functions |
|---|---|
| String manipulation | `upper`, `lower`, `trim`, `length`, `concat_ws`, `split`, `regexp_replace`, `regexp_extract` |
| Date manipulation | `to_date`, `year`, `month`, `dayofweek`, `datediff`, `date_format`, `current_date` |
| Conditional logic | `when` / `otherwise` |
| Custom logic | `udf` decorator, `StringType`, `IntegerType` |


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when,
    upper, lower, trim, length, substring,
    concat, concat_ws, split,
    regexp_extract, regexp_replace,
    to_date, year, month, dayofweek,
    datediff, date_format, current_date,
    udf
)
from pyspark.sql.types import StringType, IntegerType

spark = SparkSession.builder \
    .appName("Lesson08_EnrichingListings") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load the property listings dataset
listings = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/property_listings.csv")

print(f"Loaded {listings.count()} property listings")
print("Schema:")
listings.printSchema()

## String Functions - Cleaning Up the Address Book

Data entry is messy. Addresses might have extra spaces, inconsistent casing, or malformed ZIP codes.
PySpark's built-in string functions let you clean and transform text columns without leaving the DataFrame API.

Key functions:
- `upper(col)` / `lower(col)` — change case
- `trim(col)` — remove leading and trailing whitespace
- `length(col)` — count characters


In [ ]:
# Standardize casing and clean whitespace
cleaned = listings.withColumn("property_type_upper", upper(trim(col("property_type")))) \
    .withColumn("neighborhood_lower", lower(trim(col("neighborhood")))) \
    .withColumn("address_length", length(col("address")))

print("Casing and whitespace cleanup:")
cleaned.select(
    "property_type", "property_type_upper",
    "neighborhood", "neighborhood_lower",
    "address", "address_length"
).show(8, truncate=False)

In [ ]:
# Build a full address string by combining address + city + zip_code
# concat_ws(separator, col1, col2, ...) is the cleanest way to do this
with_full_address = listings.withColumn(
    "full_address",
    concat_ws(", ", col("address"), col("city"), col("zip_code").cast("string"))
)

# split() returns an ArrayType column; use [0] (getItem(0)) to extract the first token
# This grabs the street number from the beginning of an address like "142 Oak Street"
with_street_num = with_full_address.withColumn(
    "street_number",
    split(col("address"), " ").getItem(0)
)

print("Full address and extracted street number:")
with_street_num.select("address", "city", "zip_code", "full_address", "street_number").show(6, truncate=False)

In [ ]:
# regexp_replace: remove any non-numeric characters from zip_code
# Useful if some ZIP codes were entered as "90210-1234" (ZIP+4 format)
clean_zip = listings.withColumn(
    "zip_clean",
    regexp_replace(col("zip_code").cast("string"), "[^0-9]", "")
)

# regexp_extract: pull the street number from address using a regex
# Pattern "^(\\d+)" matches one or more digits at the start of the string
clean_zip = clean_zip.withColumn(
    "street_num_regex",
    regexp_extract(col("address"), "^(\\d+)", 1)
)

print("Cleaned ZIP codes and regex-extracted street numbers:")
clean_zip.select("address", "street_num_regex", "zip_code", "zip_clean").show(8, truncate=False)

print("\nNote: regexp_replace and regexp_extract use Java regex syntax.")
print("Backslashes must be double-escaped in Python strings: \\\\d instead of \\d")

## Date Functions - When Did This Property Hit the Market?

Dates are stored as strings in CSV files. Before you can do any date arithmetic,
you must parse them into a proper `DateType` column using `to_date()`.

Once parsed, PySpark gives you a rich set of extraction functions:

| Function | Returns |
|---|---|
| `year(col)` | Integer year (e.g. 2023) |
| `month(col)` | Integer 1–12 |
| `dayofweek(col)` | Integer 1 (Sunday) – 7 (Saturday) |
| `datediff(end, start)` | Number of days between two dates |
| `date_format(col, fmt)` | Formatted string (e.g. "March 2023") |


In [ ]:
# Convert the listing_date string to a proper DateType
# The format string must match the source data: "yyyy-MM-dd"
listings_dated = listings.withColumn(
    "listing_date_parsed",
    to_date(col("listing_date"), "yyyy-MM-dd")
)

# Extract components from the parsed date
listings_dated = listings_dated \
    .withColumn("listing_year", year(col("listing_date_parsed"))) \
    .withColumn("listing_month", month(col("listing_date_parsed"))) \
    .withColumn("listing_day_of_week", dayofweek(col("listing_date_parsed"))) \
    .withColumn("listing_month_label", date_format(col("listing_date_parsed"), "MMMM yyyy"))

print("Extracted date components from listing_date:")
listings_dated.select(
    "listing_date", "listing_date_parsed",
    "listing_year", "listing_month",
    "listing_day_of_week", "listing_month_label"
).show(8, truncate=False)

print("\ndayofweek: 1=Sunday, 2=Monday, ..., 7=Saturday")

In [ ]:
# Compute how many days each property has been on the market
# datediff(later_date, earlier_date) returns an integer number of days
listings_dom = listings_dated.withColumn(
    "days_on_market",
    datediff(current_date(), col("listing_date_parsed"))
)

print("Properties sorted by days on market (longest first):")
listings_dom.select(
    "property_id", "address", "city", "list_price",
    "listing_date", "days_on_market", "status"
).orderBy(col("days_on_market").desc()).show(10, truncate=False)

# Summarize: how long do active listings typically sit?
from pyspark.sql.functions import avg, min as spark_min, max as spark_max
print("\nDays on market summary for ACTIVE listings:")
listings_dom.filter(col("status") == "Active") \
    .agg(
        avg("days_on_market").alias("avg_days"),
        spark_min("days_on_market").alias("min_days"),
        spark_max("days_on_market").alias("max_days")
    ).show()

## Conditional Categorization with `when` / `otherwise`

The `when(condition, value).when(...).otherwise(default)` chain is PySpark's equivalent
of a SQL `CASE WHEN` statement. It's vectorized and runs entirely in the JVM — much
faster than a Python UDF.

Use it whenever you need to bucket or classify a numeric column into labeled categories.


In [ ]:
# Create an age_category column based on how old the property is
# We approximate current year as 2024 for deterministic output
current_year = 2024

listings_age = listings.withColumn(
    "property_age",
    current_year - col("year_built")
).withColumn(
    "age_category",
    when(col("property_age") < 5, "Brand New")
    .when(col("property_age") < 20, "Modern")
    .when(col("property_age") < 40, "Established")
    .otherwise("Vintage")
)

print("Property age categories:")
listings_age.select("property_id", "year_built", "property_age", "age_category").show(10)

print("\nDistribution of age categories:")
listings_age.groupBy("age_category") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [ ]:
# Create a price_category based on list_price
# Thresholds are illustrative — adjust for your local market!
listings_priced = listings_age.withColumn(
    "price_category",
    when(col("list_price") < 300000, "Budget")
    .when(col("list_price") < 600000, "Mid-Range")
    .when(col("list_price") < 1000000, "Premium")
    .otherwise("Luxury")
)

print("Price category distribution:")
listings_priced.groupBy("price_category") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

print("Price category by property type (cross-tab):")
listings_priced.groupBy("property_type", "price_category") \
    .count() \
    .orderBy("property_type", "price_category") \
    .show(20)

## User Defined Functions (UDFs) - Custom Logic

Sometimes the built-in functions aren't enough. You need custom Python logic.
That's where **UDFs (User Defined Functions)** come in.

### How UDFs work
1. You write a regular Python function
2. You wrap it with `@udf(returnType=...)` or `udf(fn, returnType)`
3. Spark serializes each row's data to Python, runs your function, and returns the result

### The catch
UDFs break Spark's internal optimization pipeline (Catalyst). Each row crosses
the JVM ↔ Python boundary twice. For large datasets this can be **10x–100x slower**
than equivalent built-in functions.

> **Rule of thumb:** Always look for a built-in function first. Use a UDF only when there
> is truly no built-in alternative.


In [ ]:
# Define a UDF that computes a composite "property score" (1-100)
# Logic: weighted combination of size, recency, and bedroom count

@udf(returnType=IntegerType())
def property_score(bedrooms, bathrooms, sqft, year_built):
    """Returns an integer score 1-100 representing overall property desirability."""
    if None in (bedrooms, bathrooms, sqft, year_built):
        return None

    # Size score: up to 40 points (cap at 4000 sqft)
    size_score = min(sqft / 4000.0, 1.0) * 40

    # Recency score: up to 30 points (newer is better; cap age at 60 years)
    age = max(0, 2024 - year_built)
    recency_score = max(0.0, 1.0 - age / 60.0) * 30

    # Bedroom score: up to 20 points (3 beds = full score)
    bed_score = min(bedrooms / 3.0, 1.0) * 20

    # Bathroom score: up to 10 points (2 baths = full score)
    bath_score = min(bathrooms / 2.0, 1.0) * 10

    total = size_score + recency_score + bed_score + bath_score
    return int(round(min(max(total, 1), 100)))


# Apply the UDF to the DataFrame
listings_scored = listings.withColumn(
    "property_score",
    property_score(
        col("bedrooms"),
        col("bathrooms"),
        col("sqft"),
        col("year_built")
    )
)

print("Properties with UDF-computed scores:")
listings_scored.select(
    "property_id", "bedrooms", "bathrooms", "sqft", "year_built", "property_score"
).orderBy(col("property_score").desc()).show(10)

print("\nScore distribution (histogram buckets):")
listings_scored.select(
    when(col("property_score") >= 80, "80-100 (Excellent)")
    .when(col("property_score") >= 60, "60-79 (Good)")
    .when(col("property_score") >= 40, "40-59 (Average)")
    .otherwise("< 40 (Below Average)").alias("score_band")
).groupBy("score_band").count().orderBy("score_band").show()

In [ ]:
# IMPORTANT: UDFs have a hidden cost.
# Let's replicate the same scoring logic using ONLY built-in functions.
# This version stays entirely in the JVM and benefits from Catalyst optimization.

from pyspark.sql.functions import least, greatest

listings_scored_builtin = listings.withColumn(
    "size_score",
    least(col("sqft") / 4000.0, lit(1.0)) * 40
).withColumn(
    "age",
    greatest(lit(0), lit(2024) - col("year_built"))
).withColumn(
    "recency_score",
    greatest(lit(0.0), lit(1.0) - col("age") / 60.0) * 30
).withColumn(
    "bed_score",
    least(col("bedrooms") / 3.0, lit(1.0)) * 20
).withColumn(
    "bath_score",
    least(col("bathrooms") / 2.0, lit(1.0)) * 10
).withColumn(
    "property_score_builtin",
    (col("size_score") + col("recency_score") + col("bed_score") + col("bath_score")).cast(IntegerType())
)

print("Built-in function equivalent (no serialization overhead):")
listings_scored_builtin.select(
    "property_id", "sqft", "year_built", "bedrooms", "bathrooms", "property_score_builtin"
).orderBy(col("property_score_builtin").desc()).show(10)

print("""
Performance comparison:
  UDF approach    : each row serialized Python <-> JVM; Catalyst cannot optimize inside UDF
  Built-in approach: runs natively in JVM; Catalyst can push down, fuse, and optimize

  For 10 million rows, built-in functions can be 10x-50x faster than UDFs.
  Reserve UDFs for logic that truly cannot be expressed with built-in functions.
""")

# Import lit - forgot to import it above, add it here
from pyspark.sql.functions import lit

## Lesson 8 Complete!

---

### What you mastered today

| Skill | Functions used |
|---|---|
| String cleaning | `upper`, `lower`, `trim`, `length` |
| String construction | `concat_ws`, `split`, `getItem` |
| Regex operations | `regexp_replace`, `regexp_extract` |
| Date parsing | `to_date`, `date_format` |
| Date arithmetic | `datediff`, `current_date`, `year`, `month`, `dayofweek` |
| Conditional columns | `when` / `otherwise` |
| Custom functions | `@udf` decorator, `StringType`, `IntegerType` |
| Performance thinking | Built-in functions preferred over UDFs |

### Key takeaways

1. **Always parse date strings** with `to_date()` before doing any date math
2. **`when().when().otherwise()`** is your go-to for categorical bucketing
3. **UDFs are powerful but costly** — exhaust built-in options first
4. **`concat_ws`** is cleaner than `concat` when you have a consistent separator
5. Regex in PySpark uses Java syntax — double-escape backslashes in Python strings

---

### Up Next: Lesson 9 - Building the Data Pipeline

Your enriched data is ready. In Lesson 9 you'll learn how to **write data to disk**,
choose the right file format (CSV vs JSON vs Parquet), partition output for fast reads,
and wrap everything into a **production-ready ETL pipeline**.

> *"A real estate agent who keeps all their listings in their head isn't very useful.
> Neither is an analysis that only lives in memory."*
